# H-010 · Half-life gate

Off versus one pre-registered half-life band `[5, 60]` (days). Freeze `HL_GATE_STAR`.

Do not open the sealed holdout in this notebook.

Next: run H-011.


## 0. Imports & Config


In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from backtest.s2_coint.report import (
    arm_selection_table,
    fold_table,
    fold_val_metrics,
    full_is_metrics,
    load_star_stack,
    median_sharpe_hint,
    plot_fold_boxplots,
    require_star,
    save_star_stack,
    write_tearsheet_pdf,
)
from backtest.s2_coint.research import (
    config_from_stack,
    is_end_for_stack,
    load_s1_weekly,
    load_s2_cuts,
    load_universe_panels,
    lookbacks_for_bar,
    n_trials_local,
    n_trials_stack,
    overlay_ols_hedge,
    overlay_kalman_hedge,
    register_hypothesis_arms,
    s2_artifacts_dir,
    s2_data_dir,
    tearsheet_path,
)
from backtest.s2_coint.runner import run_s2_backtest
from backtest.s2_coint.walkforward import embargo_bars_for_config, make_s2_folds

VERSION = "v2"
STAR_PATH = os.path.join(s2_artifacts_dir(version=VERSION, root=ROOT), "s2_star_stack.json")
LEDGER_PATH = os.path.join(os.path.dirname(STAR_PATH), "s2_variant_ledger.json")
cuts = load_s2_cuts(root=ROOT, version=VERSION)
stack = load_star_stack(STAR_PATH)
print("yahoo_drop_rate", cuts.get("yahoo_drop_rate"))
print("stack keys", sorted(stack.keys()))

HYP_ID = "H-010"
ARMS = ("off", "5_60")
require_star("HEDGE_STAR", stack.get("HEDGE_STAR"))
require_star("OVERLAP_STAR", stack.get("OVERLAP_STAR"))

require_star("POOL_STAR", stack.get("POOL_STAR"))
require_star("FDR_STAR", stack.get("FDR_STAR"))
require_star("BOOK_STAR", stack.get("BOOK_STAR"))
require_star("N_STAR", stack.get("N_STAR"))
PAIRS_STAR = list(stack.get("PAIRS_STAR") or [])
if not PAIRS_STAR:
    raise ValueError("PAIRS_STAR empty")
BAR = str(stack.get("BAR_STAR") or "1d")
train, full = load_universe_panels(None, BAR, PAIRS_STAR, root=ROOT, version=VERSION)
lb0 = lookbacks_for_bar(
    BAR,
    ols_days=stack.get("OLS_WINDOW_STAR"),
    z_days=stack.get("Z_WINDOW_STAR"),
    adf_days=stack.get("ADF_WINDOW_STAR"),
)
train = overlay_ols_hedge(train, ols_window=lb0["ols_window"], z_window=lb0["z_window"],
                          hl_window=lb0["hl_window"], adf_window=lb0["adf_window"])
full = overlay_ols_hedge(full, ols_window=lb0["ols_window"], z_window=lb0["z_window"],
                         hl_window=lb0["hl_window"], adf_window=lb0["adf_window"])
is_end = is_end_for_stack(stack, full if BAR == "1h" else train)
if BAR == "1d":
    is_panel, oos_panel = train.copy(), full.loc[pd.to_datetime(full["date"]) > is_end].copy()
else:
    from backtest.s2_coint.research import split_is_oos
    is_panel, oos_panel = split_is_oos(full, is_end=is_end)
s1_weekly = load_s1_weekly(ROOT)
print("bar", BAR, "is_end", is_end, "IS", len(is_panel), "pairs", len(PAIRS_STAR))



## 1. Walk-forward


In [ ]:
panel = is_panel.copy()
dates = pd.DatetimeIndex(pd.to_datetime(panel["date"])).sort_values().unique()
folds = make_s2_folds(dates, n_folds=3, embargo_bars=embargo_bars_for_config(bar=BAR))
display(fold_table(folds))
# Pre-registered band: 5–60 day half-life (recognized by config_from_stack / _hl_gate).
configs = {}
for arm in ARMS:
    s2 = dict(stack)
    s2["HL_GATE_STAR"] = arm
    configs[arm] = config_from_stack(s2)

register_hypothesis_arms(HYP_ID, list(configs.keys()), overwrite=True, path=LEDGER_PATH)
fold_df = fold_val_metrics(panel, folds, configs, hyp_id=HYP_ID, s1_weekly=s1_weekly)
full_is_df = full_is_metrics(panel, configs, hyp_id=HYP_ID, s1_weekly=s1_weekly)
plot_fold_boxplots(fold_df, title=f"{HYP_ID} fold-val")
plt.show()
print("median-Sharpe hint (commentary only):", median_sharpe_hint(fold_df))
display(arm_selection_table(fold_df, full_is_df))


## 2. Type `HL_GATE_STAR`


In [ ]:
HL_GATE_STAR = ""  # type after the table: "off" | "5_60"
require_star("HL_GATE_STAR", HL_GATE_STAR)
stack["HL_GATE_STAR"] = HL_GATE_STAR
save_star_stack(STAR_PATH, stack)
print("wrote", STAR_PATH, "HL_GATE_STAR", HL_GATE_STAR)


## 3. Notes for next hyp

Next: run H-011.
